# RLVR Data Playground

Use this notebook to build intuition before launching training. It starts with the Countdown verifier because Stage 1 is intentionally simple, but the sampling/splitting patterns are meant to generalize to later RLVR stages.

Kernel setup from the repo root:

```bash
uv run python -m ipykernel install --user --name posttraining --display-name "posttraining"
```

Then select the `posttraining` kernel in Jupyter.

In [ ]:
from collections import Counter
from itertools import islice
from statistics import mean

from datasets import load_dataset

from posttraining.rlvr.countdown import expression_ast_dump, verify_countdown

DATASET_NAME = "stzhao/tinyzero-countdown-data"
SEED = 0

print("imports ok")

## 1. Verifier intuition, no dataset needed

A Countdown reward needs to check the task rules, not only the final numeric value.

In [ ]:
toy_problem = {"nums": [2, 3, 4], "target": 10}

responses = {
    "valid": "<think>3 * 4 = 12, then subtract 2.</think><answer>(3 * 4) - 2</answer>",
    "wrong_target": "<answer>2 + 3 + 4</answer>",
    "reused_number": "<answer>4 + 4 + 2</answer>",
    "missing_tag": "(3 * 4) - 2",
    "unsafe": "<answer>__import__('os').system('echo nope')</answer>",
}

for name, response in responses.items():
    result = verify_countdown(response, toy_problem["nums"], toy_problem["target"])
    print(f"\n{name}")
    print("  expression:", result.expression)
    print("  value:", result.value)
    print("  answer_tag:", result.answer_tag)
    print("  parse:", result.parse)
    print("  numbers:", result.numbers)
    print("  correct:", result.correct)
    print("  error:", result.error)

In [ ]:
print(expression_ast_dump("(3 * 4) - 2"))

## 2. Load a small slice of the Countdown dataset

This cell downloads from Hugging Face the first time it runs. Keep the slice small while exploring.

In [ ]:
raw = load_dataset(DATASET_NAME, split="train")
raw

In [ ]:
sample = raw.shuffle(seed=SEED).select(range(20))
for row in sample:
    print({"nums": row["nums"], "target": row["target"]})

## 3. Profile task shape

Before training, look at problem sizes and target ranges. For Stage 1 we start with 3-number examples because they are easier to debug.

In [ ]:
profile_n = min(50_000, len(raw))
profile = raw.shuffle(seed=SEED).select(range(profile_n))

num_count_hist = Counter(len(row["nums"]) for row in profile)
targets = [int(row["target"]) for row in profile]

print("rows profiled:", profile_n)
print("num count histogram:", dict(sorted(num_count_hist.items())))
print("target min/max/mean:", min(targets), max(targets), round(mean(targets), 2))

## 4. Build a deduped train/eval split

For Countdown, order does not matter, but multiplicity does. A useful problem key is `(sorted(nums), target)`. That prevents the same problem from landing in both train and eval.

In [ ]:
def problem_key(row):
    return tuple(sorted(int(x) for x in row["nums"])), int(row["target"])


def dedupe_rows(ds, *, limit=None):
    seen = set()
    rows = []
    iterable = ds if limit is None else islice(ds, limit)
    for row in iterable:
        key = problem_key(row)
        if key in seen:
            continue
        seen.add(key)
        rows.append(row)
    return rows


three_num = raw.filter(lambda row: len(row["nums"]) == 3).shuffle(seed=SEED)
deduped = dedupe_rows(three_num, limit=20_000)

eval_size = 256
train_size = 2048
eval_rows = deduped[:eval_size]
train_rows = deduped[eval_size : eval_size + train_size]

train_keys = {problem_key(row) for row in train_rows}
eval_keys = {problem_key(row) for row in eval_rows}

print("train rows:", len(train_rows))
print("eval rows:", len(eval_rows))
print("overlap:", len(train_keys & eval_keys))
print("first train row:", {"nums": train_rows[0]["nums"], "target": train_rows[0]["target"]})

## 5. Score candidate responses against sampled rows

Use this to feel the difference between parse validity, number validity, and correctness.

In [ ]:
def score_response(row, response):
    result = verify_countdown(response, row["nums"], int(row["target"]))
    return {
        "nums": row["nums"],
        "target": int(row["target"]),
        "expression": result.expression,
        "value": str(result.value) if result.value is not None else None,
        "answer_tag": result.answer_tag,
        "parse": result.parse,
        "numbers": result.numbers,
        "correct": result.correct,
        "error": result.error,
    }


row = train_rows[0]
print({"nums": row["nums"], "target": row["target"]})

# Edit this by hand.
candidate = "<answer>1 + 2 + 3</answer>"
score_response(row, candidate)

## 6. Simulate policy attempts

Before model training, you can simulate reward aggregation with hand-written or baseline responses. Later stages can reuse this pattern by swapping in a different verifier.

In [ ]:
def aggregate_results(rows, response_fn):
    counts = Counter()
    for row in rows:
        response = response_fn(row)
        result = verify_countdown(response, row["nums"], int(row["target"]))
        counts["answer_tag"] += int(result.answer_tag)
        counts["parse"] += int(result.parse)
        counts["numbers"] += int(result.numbers)
        counts["correct"] += int(result.correct)
        counts["total"] += 1
    return {
        key: round(value / counts["total"], 3)
        for key, value in counts.items()
        if key != "total"
    }


def naive_sum(row):
    expr = " + ".join(str(x) for x in row["nums"])
    return f"<answer>{expr}</answer>"


aggregate_results(eval_rows[:100], naive_sum)

## 7. Generalize this notebook to future stages

For each RLVR stage, keep the same notebook shape:

1. Load a small slice.
2. Define a stable problem key for train/eval splitting.
3. Print examples and failure cases.
4. Run the verifier without training.
5. Aggregate verifier submetrics before looking at reward.

The key discipline is to debug the reward function and data split before launching any training run.